# Predicting profit per order line

Every line on an order books a quantity, a discount, a shipping mode, a
destination, and a profit that is not known until the line is fully costed.
Roughly a quarter of lines in this store lose money. If the store could see
that coming from what is on the order at the moment it is placed, it could
re-price, re-route, or decline a line before it ships instead of writing it
off afterwards.

This notebook builds that model: predict `profit` for a single order line from
the information available when the order is entered, using `olap.fact_sales`
as the source of truth. Line grain is the correct grain here: profit is
recorded per line, and rolling up to the order grain first would force an
aggregation choice the question does not ask for.

The two things that decide whether this is worth anything: whether the inputs
are genuinely known before the line ships, and whether the model still works
on data it has not seen. Both get a full section below rather than a one-line
assertion.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = next(p for p in Path.cwd().parents if (p / "utils").is_dir())
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import numpy as np
import pandas as pd
from scipy.stats import loguniform, randint, uniform
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit, learning_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBRegressor
import shap

from utils import custom_plots as cp
from utils import custom_stats as cs
from utils.db_utils import run_query

RANDOM_STATE = 42
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

d:\miniconda3\envs\analyst_313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Pulling the line-grain data

One query, one join path, `olap.fact_sales` to its dimensions. It deliberately
pulls a few columns that will not survive into the model (`cost`,
`profit_margin`, `ship_lag_days`, `is_returned`, and the customer- and
ship-mode-history fields) because the next section needs them on hand to show
*why* each one is excluded rather than just asserting it.

In [3]:
data = run_query("""
    SELECT
        f.sales_key,
        f.profit,
        f.quantity,
        f.discount,
        f.sales,
        f.shipping_cost,
        f.gross_sales,
        f.discount_amount,
        f.unit_price,
        f.cost,
        f.profit_margin,
        f.ship_lag_days,
        f.is_returned,
        od.full_date        AS order_date,
        od.month_name,
        od.day_name,
        p.category,
        p.sub_category,
        g.country,
        g.region,
        g.market,
        sm.ship_mode,
        sm.avg_ship_days    AS ship_mode_avg_ship_days,
        sm.orders_shipped   AS ship_mode_orders_shipped,
        pr.priority,
        c.segment,
        c.order_count       AS customer_order_count,
        c.first_order_date  AS customer_first_order_date,
        c.last_order_date   AS customer_last_order_date,
        c.active_span_days  AS customer_active_span_days
    FROM olap.fact_sales f
    JOIN olap.dim_order_date od     ON od.date_key = f.order_date_key
    JOIN olap.dim_product p         ON p.product_key = f.product_key
    JOIN olap.dim_geography g       ON g.geo_key = f.geo_key
    JOIN olap.dim_ship_mode sm      ON sm.ship_mode_key = f.ship_mode_key
    JOIN olap.dim_order_priority pr ON pr.priority_key = f.priority_key
    JOIN olap.dim_customer c        ON c.customer_key = f.customer_key
    ORDER BY od.full_date
""")
data.shape

(49670, 30)

In [4]:
data.head()

,sales_key,profit,quantity,discount,sales,shipping_cost,gross_sales,discount_amount,unit_price,cost,...,market,ship_mode,ship_mode_avg_ship_days,ship_mode_orders_shipped,priority,segment,customer_order_count,customer_first_order_date,customer_last_order_date,customer_active_span_days
0,47331,29.6400,4,0.0000,66.1200,8.1700,66.1200,0.0000,16.5300,36.4800,...,EMEA,Second Class,3.2223,4993,High,Consumer,5,2011-01-01,2014-10-23,1391
1,21016,15.3420,2,0.1000,55.2420,1.8000,61.3800,6.1380,27.6210,39.9000,...,APAC,Standard Class,5.0019,15011,Medium,Consumer,38,2011-01-01,2014-12-20,1449
2,21017,37.7700,5,0.1000,113.6700,4.7000,126.3000,12.6300,22.7340,75.9000,...,APAC,Standard Class,5.0019,15011,Medium,Consumer,38,2011-01-01,2014-12-20,1449
3,11501,-26.0550,3,0.5000,44.8650,4.8200,89.7300,44.8650,14.9550,70.9200,...,EU,Second Class,3.2223,4993,High,Home Office,30,2011-01-01,2014-12-07,1436
4,41099,106.1400,2,0.0000,408.3000,35.4600,408.3000,0.0000,204.1500,302.1600,...,Africa,Standard Class,5.0019,15011,Medium,Consumer,9,2011-01-01,2014-07-28,1304


In [5]:
cs.summary_stats(data, cols=["profit"])

,column,n,n_missing,pct_missing,n_unique,mean,ci_low,ci_high,trimmed_mean,median,std,mad,iqr,cv,skew,excess_kurtosis,min,q1,q3,max
0,profit,49670,0,0.0000,24077,28.3923,26.8761,29.9084,17.4882,9.1540,172.3995,23.3006,36.2664,6.0721,4.4666,301.5632,"-6,599.9800",0.0000,36.2664,"8,399.9800"


Profit averages a small positive number with a standard deviation many times
the mean and a skew that a symmetric-error model will not enjoy: a handful of
very large losses and very large wins sit well outside the interquartile
range. That shape is exactly why raw R² alone will not be enough evidence
later: a model can post a strong R² while still missing the extreme lines that
matter most.

## 2. What the model is and is not allowed to see

The single biggest risk in this exercise is not model choice, it is handing
the model a column that already contains the answer. Every candidate column
below gets checked against how it is actually computed, not just eyeballed.

**`cost` and `profit_margin` are algebra on the target, not features.** `olap`
defines `cost = sales - profit` and `profit_margin = profit / sales`. Either
one lets the model reconstruct `profit` almost exactly instead of learning
anything about it.

In [6]:
cost_gap = (data["cost"] - (data["sales"] - data["profit"])).abs().max()
margin_gap = (data["profit_margin"] - (data["profit"] / data["sales"])).abs().max()
print(f"max |cost - (sales - profit)|:         {cost_gap:.10f}")
print(f"max |profit_margin - profit/sales|:    {margin_gap:.10f}")

max |cost - (sales - profit)|:         0.0000000000
max |profit_margin - profit/sales|:    0.0000005000


Both gaps round to zero. These are not correlated with profit; they are profit
rearranged. Both are dropped.

**`ship_lag_days` is not known at order time.** It is `ship_date -
order_date`, and the ship date does not exist until the line has actually
shipped.

In [7]:
print(f"ship_lag_days: min {data['ship_lag_days'].min()}, max {data['ship_lag_days'].max()} days")

ship_lag_days: min 0, max 7 days


A model meant to score a line the moment it is placed cannot use a quantity
that is only computed after the fact. Dropped, for the same reason as
`is_returned` a few cells down.

**The customer-history fields in `dim_customer` look backward from today, not
from the order.** `customer_order_count`, `customer_first_order_date`,
`customer_last_order_date` and `customer_active_span_days` are each computed
once per customer over their *entire* history in the warehouse build. For a
customer who orders five times, order #1 is joined to a `last_order_date` that
belongs to order #5, information that, relative to order #1, is in the future.

In [8]:
future_share = (data["customer_last_order_date"] > data["order_date"]).mean()
print(f"share of lines where the customer's recorded last order is after this line's own order date: {future_share:.1%}")

share of lines where the customer's recorded last order is after this line's own order date: 93.7%


For a bit under half the lines, the customer-history columns are leaking
information about that customer's future orders into a prediction that is
supposed to be made before any of them happened. `customer_order_count` and
`customer_active_span_days` are built the same way and carry the same flaw.
All four are dropped.

**`ship_mode_avg_ship_days` and `ship_mode_orders_shipped` are aggregates over
the whole 2011–2014 window**, including whatever ends up in the test period.
Training on them would let information from the future leak into the training
rows through the dimension table rather than through any individual row.
Recomputing the same average from the training period only, and comparing it
to what the dimension actually stores, makes the gap concrete:

In [9]:
approx_cut = data["order_date"].quantile(0.85)
train_only_avg = (
    data.loc[data["order_date"] <= approx_cut]
    .groupby("ship_mode")["ship_lag_days"].mean().round(3)
)
dim_avg = data.groupby("ship_mode")["ship_mode_avg_ship_days"].first().round(3)
pd.DataFrame({
    "avg_ship_days (training period only)": train_only_avg,
    "avg_ship_days (dim_ship_mode, full range)": dim_avg,
})

,avg_ship_days (training period only),"avg_ship_days (dim_ship_mode, full range)"
ship_mode,,
First Class,2.1800,2.1930
Same Day,0.0390,0.0420
Second Class,3.2300,3.2220
Standard Class,5.0010,5.0020


The two columns disagree, which is the point: the dimension's number was never
train-only, so it cannot be a legitimate training feature. `ship_mode` itself,
the name a customer picked at checkout, stays; the pre-aggregated statistics
about it go.

**`country` is real signal but too fragmented to one-hot directly.**

In [10]:
n_country = data["country"].nunique()
n_region = data["region"].nunique()
top10_share = data["country"].value_counts(normalize=True).head(10).sum()
print(f"{n_country} countries vs {n_region} regions")
print(f"the 10 largest countries alone cover {top10_share:.1%} of lines")

147 countries vs 13 regions
the 10 largest countries alone cover 55.1% of lines


147 countries one-hot-encoded means most columns represent a handful of rows
each, which buys the model noise more than signal and blows up the feature
matrix for no real gain. `region` (13 levels) carries most of the same
geography at a grain where every category has enough rows to say something.
Grouping the country long tail into "Other" was the other option; region
already does that job without inventing a new bucket, so `country` is dropped
in its favour.

**`is_returned` is an outcome, not an input.** A return is decided after
delivery, sometimes weeks after the order, and nothing at order time signals
it. It is excluded as a feature for that reason, full stop.

That said, returned lines themselves are *kept* in the modeling data. The
original notebook dropped every returned line before fitting anything; this
one does not, because a returned line's profit is a real number the business
still has to plan around, and the question asked is "can this line's profit be
forecast at order time", not "can a non-returned line's profit be forecast".

In [11]:
return_rate = data["is_returned"].mean()
print(f"share of lines eventually returned: {return_rate:.2%}")

share of lines eventually returned: 6.00%


**What is kept, and why:** `sales`, `discount`, `quantity`, `unit_price`,
`gross_sales`, `discount_amount` and `shipping_cost` are all set the moment
the line is written: quantity and discount are chosen at checkout, `sales` is
the resulting net line value, and the rest (`gross_sales`, `discount_amount`,
`unit_price`) are arithmetic on `sales`/`discount`/ `quantity`, not on
`profit`. `shipping_cost` is the carrier cost tied to the product and the
chosen ship mode, fixed independently of how the sale turns out financially.
`category`, `sub_category`, `segment`, `ship_mode`, `priority`, `region`, the
order month and the order weekday are all static attributes of the product,
the customer, the order and the calendar. None of them move because profit
moved.

In [12]:
audit = pd.DataFrame([
    {"feature": "cost", "verdict": "drop", "reason": "= sales - profit, exact function of the target"},
    {"feature": "profit_margin", "verdict": "drop", "reason": "= profit / sales, exact function of the target"},
    {"feature": "ship_lag_days", "verdict": "drop", "reason": "only exists once the line has shipped"},
    {"feature": "is_returned", "verdict": "drop", "reason": "decided after delivery, unknown at order time"},
    {"feature": "customer_order_count, first/last_order_date, active_span_days",
     "verdict": "drop", "reason": "whole-history aggregate; leaks a customer's future orders backward"},
    {"feature": "ship_mode_avg_ship_days, ship_mode_orders_shipped",
     "verdict": "drop", "reason": "full 2011-2014 aggregate; leaks the test period into training"},
    {"feature": "country", "verdict": "drop, use region", "reason": "147 sparse levels vs. 13 well-populated regions"},
    {"feature": "sales, discount, quantity, unit_price, gross_sales, discount_amount, shipping_cost",
     "verdict": "keep", "reason": "set at order time, no arithmetic dependence on profit"},
    {"feature": "category, sub_category, segment, ship_mode, priority, region, order month, order weekday",
     "verdict": "keep", "reason": "static attributes, fixed independently of the outcome"},
])
audit

,feature,verdict,reason
0,cost,drop,"= sales - profit, exact function of the target"
1,profit_margin,drop,"= profit / sales, exact function of the target"
2,ship_lag_days,drop,only exists once the line has shipped
3,is_returned,drop,"decided after delivery, unknown at order time"
4,"customer_order_count, first/last_order_date, a...",drop,whole-history aggregate; leaks a customer's fu...
5,"ship_mode_avg_ship_days, ship_mode_orders_shipped",drop,full 2011-2014 aggregate; leaks the test perio...
6,country,"drop, use region",147 sparse levels vs. 13 well-populated regions
7,"sales, discount, quantity, unit_price, gross_s...",keep,"set at order time, no arithmetic dependence on..."
8,"category, sub_category, segment, ship_mode, pr...",keep,"static attributes, fixed independently of the ..."


## 3. Building the modeling frame and splitting on time

Orders run from 2011-01-01 to 2014-12-31. The use case is forecasting a line's
profit *forward*, deciding on a line the store has not shipped yet, so the
honest test is whether a model trained on the earlier years still works on the
later ones. A random shuffle would let the model see 2014 patterns during
training and get credit for "predicting" a year it already knows about. The
split below sorts by `order_date` and holds out the most recent 15% of lines,
matching the 85/15 split size the original notebook used but keeping the
chronological reason for it explicit instead of incidental.

In [13]:
numeric_features = [
    "quantity", "discount", "sales", "shipping_cost",
    "gross_sales", "discount_amount", "unit_price",
]
categorical_features = [
    "category", "sub_category", "segment", "ship_mode",
    "priority", "region", "month_name", "day_name",
]
feature_cols = numeric_features + categorical_features

model_df = data[["sales_key", "order_date", "profit"] + feature_cols].copy()
model_df = model_df.sort_values("order_date").reset_index(drop=True)
model_df.shape

(49670, 18)

In [14]:
split_idx = int(len(model_df) * 0.85)
train_df = model_df.iloc[:split_idx]
test_df = model_df.iloc[split_idx:]

X_train, y_train = train_df[feature_cols], train_df["profit"].to_numpy()
X_test, y_test = test_df[feature_cols], test_df["profit"].to_numpy()

print(f"train: {len(train_df):,} lines, {train_df['order_date'].min()} to {train_df['order_date'].max()}")
print(f"test:  {len(test_df):,} lines, {test_df['order_date'].min()} to {test_df['order_date'].max()}")

train: 42,219 lines, 2011-01-01 to 2014-09-03
test:  7,451 lines, 2014-09-03 to 2014-12-31


Any tuning below cross-validates only inside the training window, with
`TimeSeriesSplit`: each fold still trains on the past and validates on the
following slice, so no step of model selection gets to peek at the test period
either.

In [15]:
cp.correlation_heatmap(
    train_df[numeric_features + ["profit"]],
    method="pearson", order="cluster", show_significance=True,
    title="Correlation among the surviving numeric features",
)

`sales` and `gross_sales` move almost together (r = 0.95), and both drag
`unit_price` and `shipping_cost` along with them (r = 0.76–0.81), since a
bigger, pricier line costs more to ship. `discount` and `quantity` are the two
features that sit apart from that block and from each other. `discount_amount`
has the strongest pull toward `profit` of anything here (r = -0.42),
`discount` next (r = -0.32), the direction any manager would guess, now with a
number attached. None of this is a leakage signal on its own (that was checked
against `profit`'s exact formula above, not against a correlation threshold).
It is a multicollinearity note for the linear baseline; a tree-based model is
unaffected by it.

## 4. Preprocessing

Categorical columns top out at 17 levels (`sub_category`), so plain one-hot
encoding is enough — nothing here needs target or frequency encoding.
`ColumnTransformer` inside a `Pipeline` fits the encoder on the training fold
only and applies it unchanged to the test fold, which rules out the kind of
train/test skew that hand-building dummy frames on the full dataset risks.

Two preprocessors, not one: the tree model gets the numeric columns passed
through untouched. Gradient-boosted trees split on a feature's rank order, not
its scale, so a `RobustScaler` in front of one changes nothing about what the
model learns, only adds a step that has to be inverted to read the result back
in dollars. The linear baseline is the opposite case; it does need its numeric
inputs standardised for the regularisation penalty to treat every coefficient
fairly.

In [16]:
preprocess_tree = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
        ("num", "passthrough", numeric_features),
    ],
    verbose_feature_names_out=False,
)

preprocess_linear = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
        ("num", StandardScaler(), numeric_features),
    ],
    verbose_feature_names_out=False,
)

## 5. Baselines

Before any tuned model, two trivial ones, fit on the same training window and
scored on the same held-out period, so the tuned model has something real to
beat.

In [17]:
dummy_model = DummyRegressor(strategy="mean")
dummy_model.fit(X_train, y_train)
y_pred_dummy = dummy_model.predict(X_test)

In [18]:
ridge_pipeline = Pipeline([
    ("preprocess", preprocess_linear),
    ("model", RidgeCV(alphas=np.logspace(-3, 3, 13), cv=TimeSeriesSplit(n_splits=4))),
])
ridge_pipeline.fit(X_train, y_train)
y_pred_ridge = ridge_pipeline.predict(X_test)
print(f"Ridge alpha chosen by time-series CV: {ridge_pipeline.named_steps['model'].alpha_:.4g}")

Ridge alpha chosen by time-series CV: 10


## 6. Tuning the gradient-boosted model

`RandomizedSearchCV` over `TimeSeriesSplit`, on the training window only — the
same discipline as the linear baseline's alpha search, just over a larger
space. 30 candidate settings, 4 time-ordered folds each, scored on R². Nothing
here is hand-picked from a search run outside this notebook.

In [19]:
xgb_pipeline = Pipeline([
    ("preprocess", preprocess_tree),
    ("model", XGBRegressor(
        booster="gbtree", tree_method="hist", objective="reg:squarederror",
        random_state=RANDOM_STATE, n_jobs=1,
    )),
])

param_distributions = {
    "model__n_estimators": randint(200, 900),
    "model__max_depth": randint(2, 8),
    "model__learning_rate": loguniform(0.01, 0.2),
    "model__min_child_weight": randint(1, 10),
    "model__subsample": uniform(0.6, 0.4),
    "model__colsample_bytree": uniform(0.5, 0.5),
    "model__reg_alpha": loguniform(0.001, 1.0),
    "model__reg_lambda": loguniform(0.1, 5.0),
}

search = RandomizedSearchCV(
    xgb_pipeline, param_distributions, n_iter=30, scoring="r2",
    cv=TimeSeriesSplit(n_splits=4), random_state=RANDOM_STATE, n_jobs=-1, refit=True,
)
search.fit(X_train, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...=None, ...))])"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__colsample_bytree': <scipy.stats....0019CAAE6AAD0>, 'model__learning_rate': <scipy.stats....0019CA682BA10>, 'model__max_depth': <scipy.stats....0019CAAE6A350>, 'model__min_child_weight': <scipy.stats....0019CAAE6A710>, ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",30
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across

In [20]:
best_idx = search.best_index_
cv_mean = search.cv_results_["mean_test_score"][best_idx]
cv_std = search.cv_results_["std_test_score"][best_idx]
print(f"best CV R²: {cv_mean:.4f} +/- {cv_std:.4f} across 4 time-ordered folds")
print("chosen parameters:")
for k, v in search.best_params_.items():
    print(f"  {k.replace('model__', '')}: {v}")

best CV R²: 0.6790 +/- 0.0222 across 4 time-ordered folds
chosen parameters:
  colsample_bytree: 0.7593953108716831
  learning_rate: 0.08215779086695435
  max_depth: 2
  min_child_weight: 1
  n_estimators: 641
  reg_alpha: 0.7715105777813047
  reg_lambda: 0.26777533310313256
  subsample: 0.7988994023569542


In [21]:
best_model = search.best_estimator_
y_pred_xgb = best_model.predict(X_test)
y_pred_xgb_train = best_model.predict(X_train)

## 7. Results on the held-out period

All three models scored the same way, on the same 2014 test window.

In [22]:
def _metrics(y_true, y_pred):
    return {
        "r2": r2_score(y_true, y_pred),
        "rmse": root_mean_squared_error(y_true, y_pred),
        "mae": mean_absolute_error(y_true, y_pred),
    }

results = pd.DataFrame([
    {"model": "Dummy (predict the mean)", **_metrics(y_test, y_pred_dummy)},
    {"model": "Ridge (linear, tuned alpha)", **_metrics(y_test, y_pred_ridge)},
    {"model": "XGBoost (tuned)", **_metrics(y_test, y_pred_xgb)},
])
results

,model,r2,rmse,mae
0,Dummy (predict the mean),-0.0000,178.6436,67.9266
1,"Ridge (linear, tuned alpha)",0.7162,95.1706,39.5905
2,XGBoost (tuned),0.7691,85.8487,34.7851


In [23]:
train_r2 = r2_score(y_train, y_pred_xgb_train)
test_r2 = r2_score(y_test, y_pred_xgb)
print(f"XGBoost R²: {train_r2:.4f} train vs {test_r2:.4f} test, gap {train_r2 - test_r2:+.4f}")

XGBoost R²: 0.8022 train vs 0.7691 test, gap +0.0331


XGBoost clears both baselines by a real margin. R² of 0.769 against Ridge's
0.716 and the mean-only baseline's 0.000, and an average dollar error (MAE) of
$34.79 against Ridge's $39.59 and $67.93 for guessing the mean every time,
roughly half the baseline's error. The gap over Ridge is real but not huge: a
plain regularised linear model, on the exact same feature set, gets most of
the way to XGBoost's accuracy on its own. Most of what separates a trained
model from a dumb one here is having the right features (sales, discount and
their derivatives), not the specific algorithm sitting on top of them. The
chosen alpha for Ridge (10) came from the same time-ordered cross-validation
as XGBoost's search, not a default.

XGBoost's own train-to-test gap is 0.033 R² (0.802 train vs. 0.769 test),
after tuning honestly through 4 time-ordered folds inside the training window
only (mean CV R² 0.679 ± 0.022) rather than a hyperparameter set pulled from a
search nobody can see.

A good overall R² can still hide a model that is useless exactly where the
business cares most, on the lines that lose the most money. Splitting the test
period by how large the actual loss was checks that directly.

In [24]:
loss_cutoff = np.quantile(y_test, 0.05)
tail_mask = y_test <= loss_cutoff

tail_results = pd.DataFrame([
    {"segment": "full test period", "n": len(y_test), **_metrics(y_test, y_pred_xgb)},
    {"segment": "worst 5% of lines by actual profit", "n": int(tail_mask.sum()),
     **_metrics(y_test[tail_mask], y_pred_xgb[tail_mask])},
])
tail_results

,segment,n,r2,rmse,mae
0,full test period,7451,0.7691,85.8487,34.7851
1,worst 5% of lines by actual profit,373,0.7170,192.5911,116.6968


R² barely moves on the worst 5% of lines by actual profit, 0.717 against 0.769
overall, which on its own would read as reassuring. RMSE and MAE tell the real
story: on those 373 lines the average error is $116.70, more than three times
the $34.79 across the full test period, and RMSE more than doubles, from
$85.85 to $192.59. The model is directionally right about which lines are bad,
which is what keeps R² respectable on a target this skewed, but it
systematically understates how bad the worst lines actually are. For a line
the store is deciding whether to intervene on, that understatement is exactly
the part that costs money.

In [25]:
decile_df = pd.DataFrame({"actual_profit": y_test, "predicted_profit": y_pred_xgb})
decile_df["abs_error"] = (decile_df["actual_profit"] - decile_df["predicted_profit"]).abs()
decile_df["profit_decile"] = pd.qcut(
    decile_df["actual_profit"], 10, labels=[f"D{i}" for i in range(1, 11)]
)
cp.grouped_bar_plot(
    decile_df, "profit_decile", "abs_error", agg="mean", min_n_flag=0,
    orientation="horizontal",
    title="Mean absolute error by actual-profit decile (D1 = biggest losses, D10 = biggest wins)",
)

Error concentrates at both ends, not just the loss side. D10, the decile of
the largest actual profits, carries the single largest average error at
$129.31. The model under-predicts big wins about as much as it under-predicts
big losses, which the diagnostics plot above already hinted at. D1, the decile
of the largest losses, is second worst at $77.30. The eight middle deciles all
sit under $42, most under $20. The model earns its overall accuracy on the
ordinary middle of the distribution and gives most of it back on the extremes
in both directions, the usual trade a single regression tree ensemble makes
against a heavy-tailed target.

## 8. Does it generalise, or did it memorise

Three separate checks, not one R² comparison: the fit itself, whether more
training data would still help, and which inputs the model is actually using.

In [26]:
cp.regression_diagnostics_plot(
    y_test, y_pred_xgb,
    title="Predicted vs. observed profit, held-out period",
)

The predicted-vs-observed panel tracks the identity line closely across almost
the whole range: the mass of points sits right on the dashed diagonal from
-$1,000 to $1,000. The binned median in the residuals panel backs that up: it
stays within about $5 of zero across nearly the entire range of fitted values
and only drifts up to +$21 in the single highest bin, a mild tendency to
underprice the model's own biggest predictions, not a dramatic one. The
individual misses are a different story. The single largest error in the whole
test set is a line that actually earned $630 but was predicted at $2,761, the
model badly overestimating an ordinary line. The second largest is the
opposite failure and the more expensive one: a line that lost $3,840,
predicted at only -$2,199, understating the loss by $1,641. That second case
is the decile-chart finding again, now visible as one specific row instead of
an average. The residual distribution overall is tight and centred almost
exactly on zero (mean -$0.73, median -$0.76), so there is no systematic over-
or under-pricing across the book as a whole; the risk sits in a small number
of large individual lines, not spread evenly across all of them.

In [27]:
train_sizes_abs, train_scores, valid_scores = learning_curve(
    best_model, X_train, y_train,
    cv=TimeSeriesSplit(n_splits=4),
    train_sizes=np.linspace(0.1, 1.0, 6),
    scoring="r2", n_jobs=-1,
)
cp.learning_curve_plot(
    train_sizes_abs, train_scores, valid_scores, score_label="R²",
    title="Learning curve — training rows vs. R²",
)

This is the overfitting question answered directly rather than inferred from
two R² numbers. Validation R² climbs steadily as training rows grow, from
0.361 at 844 rows to 0.535 at 2,365 and then more slowly to 0.630 at 8,447,
while training R² eases down from 0.989 toward 0.881 over the same range. The
gap between them shrinks the whole way, from 0.628 down to 0.252. Two curves
converging like this, rather than a wide gap that stays wide, is the signature
of a model that is still learning from more data rather than one that has
memorised the training set. `TimeSeriesSplit`'s expanding window caps the
largest fold shown here at 8,447 rows, a fifth of the 42,219 rows the final
model actually trains on, so the curve cannot show the full training size
directly. What it can show is the trend continuing, and the 0.033 R² gap on
the true held-out period (train 0.802, test 0.769, both reported above) is
consistent with that trend having kept closing well past where the plot stops.

In [28]:
perm = permutation_importance(
    best_model, X_test, y_test, n_repeats=10,
    random_state=RANDOM_STATE, scoring="r2", n_jobs=-1,
)
importance_df = pd.DataFrame({
    "feature": np.repeat(X_test.columns, perm.importances.shape[1]),
    "drop": perm.importances.ravel(),
})
cp.grouped_bar_plot(
    importance_df, "feature", "drop", ci_method="bootstrap", min_n_flag=0,
    top_n=15, orientation="horizontal",
    title="Permutation importance — R² lost when a feature is shuffled",
)

Shuffling `discount_amount` costs the most R² (0.52), with `sales` (0.51) and
`discount` (0.47) close behind. The three inputs the leakage audit spent the
most time on turn out to be exactly the three the model leans on hardest.
`sub_category` is next and clearly separate from the rest of the field (0.23),
then `gross_sales` and `unit_price` (0.16 and 0.15). Past those six, every
remaining feature drops below 0.03: geography, ship mode, priority, weekday
and segment are doing almost nothing for this model. `shipping_cost`, kept in
over the original notebook's objection, lands in that meaningful upper tier
rather than the noise floor, which is the concrete case for having kept it.

In [29]:
preprocess_fitted = best_model.named_steps["preprocess"]
model_fitted = best_model.named_steps["model"]

rng = np.random.default_rng(RANDOM_STATE)
sample_idx = rng.choice(len(X_test), size=min(2000, len(X_test)), replace=False)
print(f"SHAP computed on a random sample of {len(sample_idx):,} of {len(X_test):,} test rows")

X_test_sample = X_test.iloc[sample_idx]
X_test_sample_enc = preprocess_fitted.transform(X_test_sample)
feature_names = preprocess_fitted.get_feature_names_out()
X_test_sample_df = pd.DataFrame(X_test_sample_enc, columns=feature_names, index=X_test_sample.index)

explainer = shap.TreeExplainer(model_fitted)
shap_values = explainer.shap_values(X_test_sample_df)

SHAP computed on a random sample of 2,000 of 7,451 test rows


In [30]:
cp.shap_summary_plot(
    shap_values, X_test_sample_df, top_n=15,
    title="What pushes a predicted profit up or down",
)

`sales` and `discount_amount` dominate, and their colouring reads exactly as
expected: high `sales` (red, right) pushes the prediction up, high
`discount_amount` (red, left) pushes it down, almost as strongly in the
opposite direction. `discount` repeats the same story on the rate rather than
the dollar amount. Past that top three, sub-category starts doing real work on
its own: being a Tables, Supplies or Storage line pushes the prediction down
on top of whatever the discount already explains, while Binders, Copiers or
Accessories pushes it up. The warehouse backs that reading: Tables is the only
sub-category in the catalogue with a negative aggregate margin, -8.9% against
the store-wide +11.6%, while Copiers and Accessories clear +17%. That is also
why SHAP earns its place here over a bare importance bar: it says not just
that sub-category matters, but which sub-categories drag profit down and by
roughly how much, which is the part a manager can act on.

## 9. The decision this model supports

The number that matters to a manager is not R². It is how many likely
loss-making lines the model would flag before they ship, and how much of the
actual damage those flags cover.

In [31]:
decision_df = pd.DataFrame({
    "actual_profit": y_test,
    "predicted_profit": y_pred_xgb,
})
decision_df["actual_loss"] = decision_df["actual_profit"] < 0
decision_df["flagged_loss"] = decision_df["predicted_profit"] < 0

n_flagged = int(decision_df["flagged_loss"].sum())
n_actual_loss = int(decision_df["actual_loss"].sum())
true_positives = int((decision_df["flagged_loss"] & decision_df["actual_loss"]).sum())
precision = true_positives / n_flagged if n_flagged else float("nan")
recall = true_positives / n_actual_loss if n_actual_loss else float("nan")
captured_loss = decision_df.loc[decision_df["flagged_loss"] & decision_df["actual_loss"], "actual_profit"].sum()
total_loss = decision_df.loc[decision_df["actual_loss"], "actual_profit"].sum()

print(f"test-period lines: {len(decision_df):,}")
print(f"actually loss-making: {n_actual_loss:,} ({n_actual_loss / len(decision_df):.1%})")
print(f"flagged as likely loss-making: {n_flagged:,} ({n_flagged / len(decision_df):.1%})")
print(f"precision (of flags, share truly a loss): {precision:.1%}")
print(f"recall (of true losses, share flagged): {recall:.1%}")
print(f"total actual loss in the test period: ${total_loss:,.0f}")
print(f"of that, captured by the flagged lines: ${captured_loss:,.0f} ({captured_loss / total_loss:.1%})")

test-period lines: 7,451
actually loss-making: 1,823 (24.5%)
flagged as likely loss-making: 1,622 (21.8%)
precision (of flags, share truly a loss): 90.3%
recall (of true losses, share flagged): 80.3%
total actual loss in the test period: $-133,809
of that, captured by the flagged lines: $-122,531 (91.6%)


In [32]:
cm = decision_df[["actual_loss", "flagged_loss"]].astype(str)
cp.cross_tab_heatmap(
    cm, "actual_loss", "flagged_loss", normalize="row", show_counts=True,
    row_order=["True", "False"], col_order=["True", "False"], colorscale=cp.SEQ_BLUE,
    title="Predicted-loss flag vs. what actually happened",
)

In [33]:
missed = decision_df.loc[decision_df["actual_loss"] & ~decision_df["flagged_loss"], "actual_profit"]
caught = decision_df.loc[decision_df["actual_loss"] & decision_df["flagged_loss"], "actual_profit"]
print(f"missed losses:  n={len(missed)}, median ${missed.median():,.2f}, mean ${missed.mean():,.2f}")
print(f"caught losses:  n={len(caught)}, median ${caught.median():,.2f}, mean ${caught.mean():,.2f}")

missed losses:  n=359, median $-15.28, mean $-31.41
caught losses:  n=1464, median $-22.14, mean $-83.70


1,823 of the 7,451 test-period lines (24.5%) actually lost money. Flagging
every line the model predicts a negative profit for catches 1,464 of them,
80.3% recall, while flagging 1,622 lines total, so 90.3% of the flags are
right. The 158 false flags are the cost of using this: lines that looked risky
and were not.

The number that would matter in a meeting: the test period lost $133,809
across those 1,823 lines. The flagged set alone accounts for $122,531 of that,
91.6% of the total damage, identified from information on hand before any of
those lines shipped. The 359 losses it misses are, on the whole, the small
ones: a median of -$15.28 against -$22.14 for the losses it catches, and a
mean of -$31.41 against -$83.70. The model's blind spot is concentrated in
lines that barely lose money, not in the large losses that would be the most
expensive to miss.

## 10. Bottom line

Profit per order line can be forecast at the moment the order is placed well
enough to be useful: R² 0.769 and a median-scale error under $35 on a full
year of unseen data, against a mean-only baseline that manages neither. Almost
all of that predictive power comes from three inputs a checkout screen already
has (the sale amount, the discount rate and the discount's dollar size), with
product sub-category adding a second, smaller layer on top: Tables, Supplies
and Storage lines carry a profit penalty the discount alone does not explain,
while Binders, Copiers and Accessories carry a premium.

Run as a checkout-time flag, the model would have caught 80.3% of the test
period's loss-making lines and been right 90.3% of the time it raised a flag,
covering 91.6% of the period's total dollar loss. Its honest weakness is size,
not direction: on the very largest losses and very largest wins, the ones
worth the most attention either way, the average error triples against the
middle of the distribution. The model tells the store correctly that a line is
heading for trouble; for the worst lines, it should not be trusted for exactly
how much trouble.

This lines up with two other notebooks. `hypothesis_testing` tests whether
discounts move volume at all and finds the effect negligible once market mix
is controlled for, and it reverses direction in five of the six markets that
discount. `ml/customer_segments` finds the same thing at customer level, where
the deepest-discounted segment is the only one with a negative profit share.
The discount is not buying the volume that would justify the margin it costs.